In [1]:
import h5py
import numpy as np
from tqdm import tqdm
import os
import subprocess

In [16]:
input_file = "mva-dlmi-2025-histopathology-ood-classification/val.h5"
output_prefix = "data/val/val_"
num_splits = 1000

In [18]:
with h5py.File(input_file, "r") as f:
    indices = list(f.keys())  # Liste des indices des images
    total_images = len(indices)
    split_size = total_images // num_splits  # Taille d'un sous-ensemble
    remainder = total_images % num_splits  # Reste des images

    # Diviser les indices en plusieurs groupes
    split_indices = [indices[i * split_size: (i + 1) * split_size] for i in range(num_splits)]
    for i in range(remainder):
        split_indices[i].append(indices[num_splits * split_size + i])

    # Sauvegarde dans plusieurs fichiers
    for i, subset in tqdm(enumerate(split_indices)):
        output_file = f"{output_prefix}{i}.h5"
        with h5py.File(output_file, "w") as out_f:
            for idx in subset:
                group = out_f.create_group(idx)
                group.create_dataset("img", data=f[idx]["img"][:])  # Copier l'image
                group.create_dataset("label", data=f[idx]["label"][()])  # Copier le label
                group.create_dataset("metadata", data=f[idx]["metadata"][()])  # Copier les métadonnées

1000it [02:33,  6.53it/s]


In [6]:
batch_size = 1  # Nombre de fichiers par commit
counter = 0
files_to_add = []

# Lister tous les fichiers .h5 dans train, test et val
for root, _, files in os.walk("data"):
    for file in files:
        if file.endswith(".h5"):
            files_to_add.append(os.path.join(root, file))

In [7]:
# Découper la liste en paquets et pousser en plusieurs fois
for i in tqdm(range(0, len(files_to_add), batch_size)):
    batch = files_to_add[i : i + batch_size]

    # Ajouter les fichiers au commit
    subprocess.run(["git", "add"] + batch)
    subprocess.run(["git", "commit", "-m", f"Ajout de {len(batch)} fichiers"])
    subprocess.run(["git", "push", "origin", "main"])  # Remplace "main" si besoin

100%|██████████| 3000/3000 [6:02:25<00:00,  7.25s/it]      
